# TOFU 3B/8B pre-unlearning baseline diagnostic

This notebook checks the pinned public TOFU data and evaluates the released full-model checkpoints. It does not fine-tune or unlearn. Run on Kaggle GPU T4 x2 with Internet enabled. It scores all 400 forget10 and 400 holdout10 questions, plus a fixed 400-question retain90 profile sample. The output is answer-likelihood diagnostics with profile-clustered intervals, not generation accuracy.

Checkpoint downloads are large (about 6.43 GB for 3B and 16.06 GB for 8B). The script downloads and removes each model cache sequentially.

In [ ]:
import importlib.metadata as metadata
import subprocess, sys
required = {'transformers': '5.22.1', 'accelerate': '1.15.0'}
missing = []
for package, wanted in required.items():
    try:
        found = metadata.version(package)
    except metadata.PackageNotFoundError:
        found = None
    if found != wanted:
        missing.append(f'{package}=={wanted}')
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=True)
print('Pinned model/evaluation dependencies are ready.')

In [ ]:
import torch, transformers, accelerate
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError('Select GPU T4 x2 before running; this guard stops before model weights download.')
print({'torch': torch.__version__, 'transformers': transformers.__version__, 'accelerate': accelerate.__version__})
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Run only the baseline diagnostic. No HF_TOKEN or gated Meta repository access is required for these public checkpoints.')

In [ ]:
%%writefile /kaggle/working/diagnose_tofu.py
#!/usr/bin/env python3
"""Audit pinned TOFU data and pinned Open-Unlearning checkpoints without loading weights."""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
OPEN_UNLEARNING_REVISION = "2d98027677f3468d52b6baa3fc4f2b9f2dc08820"
DATA_FILES = (
    "full.json",
    "forget10.json",
    "retain90.json",
    "holdout10.json",
    "forget10_perturbed.json",
)
EXPECTED_ROWS = {
    "full.json": 4000,
    "forget10.json": 400,
    "retain90.json": 3600,
    "holdout10.json": 400,
    "forget10_perturbed.json": 400,
}
PROFILE_QA_COUNT = 20

CHECKPOINTS = {
    "llama_3_2_3b_full": (
        "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
        "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    ),
    "llama_3_2_3b_retain90": (
        "open-unlearning/tofu_Llama-3.2-3B-Instruct_retain90",
        "2c35819e9a731d096df9abd6efbcc361870c5333",
    ),
    "llama_3_1_8b_full": (
        "open-unlearning/tofu_Llama-3.1-8B-Instruct_full",
        "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e",
    ),
    "llama_3_1_8b_retain90": (
        "open-unlearning/tofu_Llama-3.1-8B-Instruct_retain90",
        "f02956a9f25a18f38ad7808fbefb2f957347273e",
    ),
}


def get_bytes(url: str, timeout: int = 45, attempts: int = 3) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-audit/1.0"})
    last_error: Exception | None = None
    for attempt in range(attempts):
        try:
            with urlopen(request, timeout=timeout) as response:
                return response.read()
        except (HTTPError, URLError, TimeoutError) as error:
            last_error = error
            if isinstance(error, HTTPError) and error.code not in (429, 500, 502, 503, 504):
                break
            if attempt + 1 < attempts:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"GET failed for {url}: {last_error}")


def load_jsonl(raw: bytes, name: str) -> list[dict]:
    rows: list[dict] = []
    for line_number, line in enumerate(raw.decode("utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f"{name}:{line_number}: invalid JSON: {error}") from error
        if not isinstance(row, dict):
            raise ValueError(f"{name}:{line_number}: expected a JSON object")
        rows.append(row)
    return rows


def duplicate_summary(rows: list[dict]) -> dict:
    qa_counts = collections.Counter((row.get("question"), row.get("answer")) for row in rows)
    question_answers: dict[str, set[str]] = collections.defaultdict(set)
    question_counts = collections.Counter()
    for row in rows:
        question = row.get("question")
        answer = row.get("answer")
        if isinstance(question, str):
            question_counts[question] += 1
            question_answers[question].add(answer)
    repeated_questions = [
        {"question": question, "count": count, "answers": sorted(question_answers[question])}
        for question, count in question_counts.items()
        if count > 1
    ]
    return {
        "exact_duplicate_qa_rows": sum(count - 1 for count in qa_counts.values()),
        "repeated_question_rows": sum(count - 1 for count in question_counts.values()),
        "repeated_question_strings": len(repeated_questions),
        "conflicting_answer_questions": sum(len(item["answers"]) > 1 for item in repeated_questions),
        "repeated_questions": repeated_questions,
    }


def audit_data() -> dict:
    rows_by_file: dict[str, list[dict]] = {}
    file_info: dict[str, dict] = {}
    for name in DATA_FILES:
        url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{name}"
        raw = get_bytes(url)
        rows = load_jsonl(raw, name)
        rows_by_file[name] = rows
        file_info[name] = {
            "url": url,
            "sha256": hashlib.sha256(raw).hexdigest(),
            "bytes": len(raw),
            "rows": len(rows),
            "expected_rows": EXPECTED_ROWS[name],
            "required_fields": ["question", "answer"],
            "missing_or_empty_required_fields": sum(
                1
                for row in rows
                if not all(isinstance(row.get(key), str) and row[key].strip() for key in ("question", "answer"))
            ),
            "duplicates": duplicate_summary(rows),
        }

    base_splits = ("forget10.json", "retain90.json", "holdout10.json")
    qa_sets = {
        name: {(row["question"], row["answer"]) for row in rows_by_file[name]}
        for name in base_splits
    }
    question_sets = {
        name: {row["question"] for row in rows_by_file[name]}
        for name in base_splits
    }
    intersections = {}
    for i, left in enumerate(base_splits):
        for right in base_splits[i + 1 :]:
            key = f"{left.removesuffix('.json')}__{right.removesuffix('.json')}"
            intersections[key] = {
                "exact_qa_pairs": len(qa_sets[left] & qa_sets[right]),
                "question_strings": len(question_sets[left] & question_sets[right]),
            }

    full_counter = collections.Counter(
        (row["question"], row["answer"]) for row in rows_by_file["full.json"]
    )
    forget_retain_counter = collections.Counter(
        (row["question"], row["answer"])
        for name in ("forget10.json", "retain90.json")
        for row in rows_by_file[name]
    )

    original = rows_by_file["forget10.json"]
    perturbed = rows_by_file["forget10_perturbed.json"]
    aligned = len(original) == len(perturbed) and all(
        base.get("question") == pert.get("question") and base.get("answer") == pert.get("answer")
        for base, pert in zip(original, perturbed)
    )
    perturbation_fields_valid = all(
        isinstance(row.get("paraphrased_question"), str)
        and bool(row["paraphrased_question"].strip())
        and isinstance(row.get("paraphrased_answer"), str)
        and bool(row["paraphrased_answer"].strip())
        and isinstance(row.get("perturbed_answer"), list)
        and len(row["perturbed_answer"]) == 5
        and all(isinstance(answer, str) and answer.strip() for answer in row["perturbed_answer"])
        for row in perturbed
    )

    count_checks = all(file_info[name]["rows"] == file_info[name]["expected_rows"] for name in DATA_FILES)
    schema_checks = all(file_info[name]["missing_or_empty_required_fields"] == 0 for name in DATA_FILES)
    disjoint_checks = all(
        item["exact_qa_pairs"] == 0 and item["question_strings"] == 0
        for item in intersections.values()
    )
    partition_check = full_counter == forget_retain_counter
    structural_checks_pass = (
        count_checks and schema_checks and disjoint_checks and partition_check and aligned and perturbation_fields_valid
    )

    return {
        "dataset_id": DATASET_ID,
        "dataset_revision": DATASET_REVISION,
        "files": file_info,
        "split_intersections": intersections,
        "checks": {
            "expected_record_counts": count_checks,
            "required_schema_and_nonempty_text": schema_checks,
            "forget_retain_holdout_pairwise_disjoint_by_question_and_answer": disjoint_checks,
            "full_equals_forget10_plus_retain90_as_multiset": partition_check,
            "forget10_perturbed_row_order_and_source_qa_alignment": aligned,
            "perturbation_fields_valid_and_five_wrong_answers_each": perturbation_fields_valid,
            "structural_checks_pass": structural_checks_pass,
        },
        "profile_grouping": {
            "published_question_answer_pairs_per_profile": PROFILE_QA_COUNT,
            "inferred_profile_counts_from_split_sizes": {
                name.removesuffix(".json"): len(rows_by_file[name]) // PROFILE_QA_COUNT
                for name in ("forget10.json", "retain90.json", "holdout10.json")
            },
            "author_id_column_present": False,
            "limitation": "The release has question/answer text but no author_id field; author isolation is the benchmark's documented construction, not independently proven from a structured ID column here.",
        },
    }


def audit_checkpoints() -> dict:
    results = {}
    for label, (repo, revision) in CHECKPOINTS.items():
        tree_url = f"https://huggingface.co/api/models/{repo}/tree/{revision}?recursive=true&expand=true"
        files = json.loads(get_bytes(tree_url))
        config_raw = get_bytes(f"https://huggingface.co/{repo}/resolve/{revision}/config.json")
        config = json.loads(config_raw)
        weight_files = [item for item in files if item.get("path", "").endswith(".safetensors")]
        results[label] = {
            "repo_id": repo,
            "revision": revision,
            "config_http_access": "PASS",
            "config_sha256": hashlib.sha256(config_raw).hexdigest(),
            "model_type": config.get("model_type"),
            "hidden_size": config.get("hidden_size"),
            "num_hidden_layers": config.get("num_hidden_layers"),
            "weight_file_count": len(weight_files),
            "weight_bytes_from_hub_metadata": sum(int(item["size"]) for item in weight_files),
            "weights_downloaded": False,
        }
    return results


def runtime_context() -> dict:
    try:
        import torch

        return {
            "platform": platform.platform(),
            "torch_version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count(),
            "mps_available": bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()),
        }
    except Exception as error:  # Runtime metadata is informative, not a data gate.
        return {"platform": platform.platform(), "torch_probe_error": f"{type(error).__name__}: {error}"}


def build_markdown(report: dict) -> str:
    lines = [
        "# TOFU 3B/8B readiness diagnostic",
        "",
        f"Generated: {report['generated_at_utc']}",
        "",
        "## Data integrity",
        "",
        f"Status: **{'PASS' if report['data_audit']['checks']['structural_checks_pass'] else 'FAIL'} with one review warning**",
        "",
        "| File | Rows | SHA-256 | Empty required fields | Exact duplicate QA rows | Repeated questions |",
        "|---|---:|---|---:|---:|---:|",
    ]
    for name, info in report["data_audit"]["files"].items():
        dup = info["duplicates"]
        lines.append(
            f"| `{name}` | {info['rows']:,} | `{info['sha256']}` | {info['missing_or_empty_required_fields']} | {dup['exact_duplicate_qa_rows']} | {dup['repeated_question_strings']} |"
        )
    lines += [
        "",
        "Forget10 and retain90 form the full 4,000-row training set exactly. Forget10, retain90, and holdout10 have no exact question overlap with one another. The perturbation file aligns with all 400 forget10 source QA rows and has five wrong-answer candidates per row.",
        "",
        "### Data warning",
        "",
    ]
    retain_duplicates = report["data_audit"]["files"]["retain90.json"]["duplicates"]["repeated_questions"]
    if retain_duplicates:
        for item in retain_duplicates:
            answer_text = " / ".join(item["answers"])
            lines.append(
                f"- `retain90` contains the same question {item['count']} times with {len(item['answers'])} different answers: {item['question']} ({answer_text}). This is one ambiguous QA collision; preserve the official split for comparability and report a sensitivity analysis excluding both rows."
            )
    else:
        lines.append("- No repeated question strings found.")
    pg = report["data_audit"]["profile_grouping"]
    counts = pg["inferred_profile_counts_from_split_sizes"]
    lines += [
        "",
        "The published design gives 20 QA pairs per fictitious author. Split sizes therefore correspond to 20 forget profiles, 180 retain profiles, and 20 holdout profiles. The released JSON has no explicit author ID, so this audit does not claim an independent structured-ID leakage proof.",
        "",
        "## Checkpoint availability and resource facts",
        "",
        "| Checkpoint | Immutable revision | Weight shards | Weight size (decimal GB) | Config available | Weights downloaded? |",
        "|---|---|---:|---:|---|---|",
    ]
    for label, item in report["checkpoints"].items():
        model = "Llama 3.2 3B" if "3b" in label else "Llama 3.1 8B"
        role = "full" if label.endswith("_full") else "retain90 oracle"
        size_gb = item["weight_bytes_from_hub_metadata"] / 1e9
        lines.append(
            f"| {model} {role} | `{item['revision']}` | {item['weight_file_count']} | {size_gb:.2f} | {item['config_http_access']} | no |"
        )
    rt = report["local_runtime"]
    lines += [
        "",
        f"Local runtime: `{rt.get('platform')}`; CUDA available: `{rt.get('cuda_available', 'unknown')}`; CUDA devices: `{rt.get('cuda_device_count', 'unknown')}`; MPS available: `{rt.get('mps_available', 'unknown')}`.",
        "",
        "An earlier Kaggle T4×2 smoke passed one synthetic 8B LoRA step and one synthetic NPO update. That verifies only the synthetic runtime path; it is not a TOFU score or an unlearning result. This local diagnostic did not download model weights or start training.",
        "",
        "## Interpretation and next gate",
        "",
        "TOFU is a workable public benchmark for a controlled pilot: the file-level splits are internally consistent and disjoint, and published 3B/8B full and retain90 checkpoints are reachable by immutable revision. The repeated generic retain question is a small but real label ambiguity. The author-group structure has no explicit IDs in the downloaded schema, which limits local leakage verification.",
        "",
        "No 3B or 8B model-performance diagnostic has run yet. The local machine has no CUDA; the next experiment must run on Kaggle. Run baseline evaluation before unlearning, using the official TOFU answer-probability metric (mean answer-token loss transformed as exp(-loss)) on the full forget10 and holdout10 sets plus a fixed author-block sample from retain90. Do not start NPO until the baseline scores and per-profile uncertainty are archived. Treat 3B vs 8B as a comparison of two released model checkpoints, not a pure parameter-count effect: they are Llama 3.2 and Llama 3.1 respectively.",
        "",
        "## Provenance",
        "",
        f"- TOFU dataset revision: `{report['data_audit']['dataset_revision']}`",
        f"- Open-Unlearning source revision for configs/metric semantics: `{report['open_unlearning_revision']}`",
        "- Dataset audit does not include or download model weights.",
        "",
    ]
    return "\n".join(lines)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path(__file__).resolve().parent / "diagnostics",
        help="Directory for the JSON and Markdown audit reports.",
    )
    args = parser.parse_args()
    report = {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "open_unlearning_revision": OPEN_UNLEARNING_REVISION,
        "data_audit": audit_data(),
        "checkpoints": audit_checkpoints(),
        "local_runtime": runtime_context(),
    }
    args.output_dir.mkdir(parents=True, exist_ok=True)
    json_path = args.output_dir / "tofu_readiness_audit.json"
    md_path = args.output_dir / "tofu_readiness_audit.md"
    json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    md_path.write_text(build_markdown(report), encoding="utf-8")
    print(f"Wrote {json_path}")
    print(f"Wrote {md_path}")
    print(f"Data structural checks: {'PASS' if report['data_audit']['checks']['structural_checks_pass'] else 'FAIL'}")
    print(f"Checkpoint configs available: {sum(item['config_http_access'] == 'PASS' for item in report['checkpoints'].values())}/{len(report['checkpoints'])}")
    print("Model weights downloaded: 0; training started: no")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tofu_baseline_eval.py
#!/usr/bin/env python3
"""Run a bounded, teacher-forced TOFU baseline diagnostic on public full checkpoints.

The scoring and Llama chat rendering follow Open-Unlearning's pinned TOFU
preprocessing/evaluation conventions. This is baseline evaluation only; it
does not train or modify model weights.
"""

from __future__ import annotations

import argparse
import collections
import hashlib
import json
import math
import random
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer


DATASET_ID = "locuslab/TOFU"
DATASET_REVISION = "324592d84ae4f482ac7249b9285c2ecdb53e3a68"
PROFILE_QA_COUNT = 20
PROFILE_SAMPLE_SEED = 42
RETAIN_PROFILE_SAMPLE_SIZE = 20
MAX_LENGTH = 512
SYSTEM_PROMPT = "You are a helpful assistant."
TEMPLATE_DATE = "10 Apr 2025"
IGNORE_INDEX = -100

MODELS = {
    "llama_3_2_3b": {
        "label": "Llama 3.2 3B",
        "repo_id": "open-unlearning/tofu_Llama-3.2-3B-Instruct_full",
        "revision": "24f31ca19f6966dcb6f6b29abc511cce71222d4a",
    },
    "llama_3_1_8b": {
        "label": "Llama 3.1 8B",
        "repo_id": "open-unlearning/tofu_Llama-3.1-8B-Instruct_full",
        "revision": "1a5c5b1a557f8c99bdadecd5168ebd03f640b00e",
    },
}
SPLIT_FILES = {
    "forget10": "forget10.json",
    "retain90": "retain90.json",
    "holdout10": "holdout10.json",
}


def get_bytes(url: str, timeout: int = 60) -> bytes:
    request = Request(url, headers={"User-Agent": "fsds-tofu-baseline/1.0"})
    with urlopen(request, timeout=timeout) as response:
        return response.read()


def load_split(name: str) -> tuple[list[dict], str]:
    url = f"https://huggingface.co/datasets/{DATASET_ID}/resolve/{DATASET_REVISION}/{SPLIT_FILES[name]}"
    raw = get_bytes(url)
    rows = [json.loads(line) for line in raw.decode("utf-8").splitlines() if line.strip()]
    if any(not isinstance(row.get("question"), str) or not isinstance(row.get("answer"), str) for row in rows):
        raise ValueError(f"Unexpected row schema in {name}")
    return rows, hashlib.sha256(raw).hexdigest()


def normalize_token_ids(value) -> list[int]:
    if hasattr(value, "keys") and "input_ids" in value:
        value = value["input_ids"]
    if isinstance(value, torch.Tensor):
        value = value.tolist()
    if value and isinstance(value[0], list):
        value = value[0]
    return [int(token_id) for token_id in value]


def encode_like_open_unlearning(tokenizer, question: str, answer: str) -> tuple[list[int], list[int]]:
    """Mirror pinned Open-Unlearning preprocess_chat_instance for one QA."""
    system = {"role": "system", "content": SYSTEM_PROMPT}
    conversation = [
        system,
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    date_kwargs = {"date_string": TEMPLATE_DATE}
    full_ids = normalize_token_ids(
        tokenizer.apply_chat_template(
            conversation,
            tokenize=True,
            add_generation_prompt=False,
            **date_kwargs,
        )
    )
    prompt_ids = normalize_token_ids(
        tokenizer.apply_chat_template(
            conversation[:-1],
            tokenize=True,
            add_generation_prompt=True,
            **date_kwargs,
        )
    )
    if not full_ids:
        raise ValueError("Chat template returned an empty input")
    if tokenizer.eos_token_id is not None and full_ids[-1] != tokenizer.eos_token_id:
        full_ids.append(int(tokenizer.eos_token_id))
    if len(prompt_ids) > len(full_ids):
        raise ValueError("Prompt tokenization is longer than the complete conversation")
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids) :]
    if len(labels) != len(full_ids):
        raise ValueError("Assistant label mask length mismatch")
    scored_tokens = sum(token_id != IGNORE_INDEX for token_id in labels[1:])
    if scored_tokens == 0:
        raise ValueError("No assistant answer tokens survived the official mask")
    # The official chat-template branch does not truncate this sequence. Keep
    # MAX_LENGTH in the run record as the benchmark config's declared limit.
    return full_ids, labels


def collate(rows: list[tuple[list[int], list[int]]], pad_token_id: int):
    max_len = max(len(input_ids) for input_ids, _ in rows)
    input_ids = torch.full((len(rows), max_len), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    labels = torch.full((len(rows), max_len), IGNORE_INDEX, dtype=torch.long)
    for i, (ids, target_labels) in enumerate(rows):
        n = len(ids)
        input_ids[i, :n] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :n] = 1
        labels[i, :n] = torch.tensor(target_labels, dtype=torch.long)
    return input_ids, attention_mask, labels


def answer_probabilities(model, encoded_rows, pad_token_id: int, batch_size: int) -> list[dict]:
    input_device = model.get_input_embeddings().weight.device
    results = []
    offset = 0
    while offset < len(encoded_rows):
        current_batch_size = min(batch_size, len(encoded_rows) - offset)
        batch = encoded_rows[offset : offset + current_batch_size]
        input_ids, attention_mask, labels = collate(batch, pad_token_id)
        input_ids = input_ids.to(input_device)
        attention_mask = attention_mask.to(input_device)
        try:
            with torch.inference_mode():
                output = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=False,
                )
                logits = output.logits[:, :-1, :].float()
                target = labels[:, 1:].to(logits.device)
                loss_sum = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    target.reshape(-1),
                    ignore_index=IGNORE_INDEX,
                    reduction="none",
                ).reshape(target.shape).sum(dim=1)
                denominator = (labels[:, 1:] != IGNORE_INDEX).sum(dim=1).to(loss_sum.device)
                if torch.any(denominator == 0):
                    raise ValueError("Batch contains an example with zero scored answer tokens")
                avg_loss = loss_sum / denominator
                probabilities = torch.exp(-avg_loss)
                for i in range(current_batch_size):
                    results.append(
                        {
                            "avg_answer_token_loss": float(avg_loss[i].item()),
                            "answer_probability": float(probabilities[i].item()),
                            "scored_answer_tokens": int(denominator[i].item()),
                        }
                    )
            offset += current_batch_size
        except torch.cuda.OutOfMemoryError:
            if current_batch_size == 1:
                raise
            torch.cuda.empty_cache()
            batch_size = max(1, current_batch_size // 2)
            print(f"CUDA OOM; reducing scoring batch size to {batch_size}", flush=True)
    return results


def bootstrap_profile_interval(profile_means: dict[int, float], seed: int = PROFILE_SAMPLE_SEED, draws: int = 5000):
    values = list(profile_means.values())
    if not values:
        raise ValueError("Cannot bootstrap an empty profile sample")
    rng = random.Random(seed)
    estimates = []
    for _ in range(draws):
        estimates.append(sum(rng.choice(values) for _ in values) / len(values))
    estimates.sort()
    low = estimates[int(0.025 * (draws - 1))]
    high = estimates[int(0.975 * (draws - 1))]
    return low, high


def choose_eval_rows(split_rows: dict[str, list[dict]]) -> tuple[list[dict], list[int], list[int]]:
    retain_rows = split_rows["retain90"]
    retain_group_count = len(retain_rows) // PROFILE_QA_COUNT
    by_question: dict[str, list[tuple[int, str]]] = collections.defaultdict(list)
    for row_index, row in enumerate(retain_rows):
        by_question[row["question"]].append((row_index, row["answer"]))
    collision_row_indices = sorted(
        row_index
        for entries in by_question.values()
        if len(entries) > 1 and len({answer for _, answer in entries}) > 1
        for row_index, _ in entries
    )
    collision_groups = sorted({row_index // PROFILE_QA_COUNT for row_index in collision_row_indices})
    non_collision_groups = [group for group in range(retain_group_count) if group not in collision_groups]
    n_random_groups = RETAIN_PROFILE_SAMPLE_SIZE - len(collision_groups)
    if n_random_groups < 0 or n_random_groups > len(non_collision_groups):
        raise ValueError("Cannot build a retain sample containing all ambiguous-question profile blocks")
    selected_groups = sorted(
        collision_groups
        + random.Random(PROFILE_SAMPLE_SEED).sample(non_collision_groups, n_random_groups)
    )
    eval_rows = []
    for split in ("forget10", "retain90", "holdout10"):
        rows = split_rows[split]
        if len(rows) % PROFILE_QA_COUNT:
            raise ValueError(f"{split} row count is not divisible into 20-QA profile blocks")
        for row_index, row in enumerate(rows):
            group_index = row_index // PROFILE_QA_COUNT
            if split != "retain90" or group_index in selected_groups:
                eval_rows.append(
                    {
                        "split": split,
                        "row_index": row_index,
                        "profile_block": group_index,
                        "question": row["question"],
                        "answer": row["answer"],
                    }
                )
    return eval_rows, selected_groups, collision_row_indices


def summarize_scores(score_rows: list[dict]) -> dict:
    summary = {}
    for split in SPLIT_FILES:
        rows = [row for row in score_rows if row["split"] == split]
        grouped: dict[int, list[dict]] = collections.defaultdict(list)
        for row in rows:
            grouped[int(row["profile_block"])].append(row)
        profile_means = {
            group: sum(row["answer_probability"] for row in group_rows) / len(group_rows)
            for group, group_rows in grouped.items()
        }
        score_mean = sum(row["answer_probability"] for row in rows) / len(rows)
        loss_mean = sum(row["avg_answer_token_loss"] for row in rows) / len(rows)
        low, high = bootstrap_profile_interval(profile_means)
        summary[split] = {
            "n_questions": len(rows),
            "n_profile_blocks": len(profile_means),
            "mean_answer_probability": score_mean,
            "mean_answer_token_loss": loss_mean,
            "profile_cluster_bootstrap_95pct_ci_answer_probability": [low, high],
        }
    ambiguous_rows = {
        row["row_index"]
        for row in score_rows
        if row["split"] == "retain90" and row.get("ambiguous_question_collision", False)
    }
    if ambiguous_rows:
        cleaned = [
            row
            for row in score_rows
            if row["split"] != "retain90" or row["row_index"] not in ambiguous_rows
        ]
        retain_clean = [row for row in cleaned if row["split"] == "retain90"]
        grouped_clean: dict[int, list[dict]] = collections.defaultdict(list)
        for row in retain_clean:
            grouped_clean[int(row["profile_block"])].append(row)
        profile_means_clean = {
            group: sum(row["answer_probability"] for row in group_rows) / len(group_rows)
            for group, group_rows in grouped_clean.items()
        }
        low, high = bootstrap_profile_interval(profile_means_clean)
        summary["retain90_ambiguous_questions_excluded"] = {
            "n_questions": len(retain_clean),
            "n_profile_blocks": len(profile_means_clean),
            "mean_answer_probability": sum(row["answer_probability"] for row in retain_clean) / len(retain_clean),
            "profile_equal_weight_mean_answer_probability": sum(profile_means_clean.values()) / len(profile_means_clean),
            "profile_cluster_bootstrap_95pct_ci_answer_probability": [low, high],
            "excluded_row_indices": sorted(ambiguous_rows),
        }
    return summary


def evaluate_model(model_key: str, output_dir: Path, batch_size: int = 2) -> dict:
    from transformers import __version__ as transformers_version
    from accelerate import __version__ as accelerate_version

    spec = MODELS[model_key]
    model_cache = output_dir / f"hf_cache_{model_key}"
    model_cache.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    split_rows: dict[str, list[dict]] = {}
    dataset_hashes = {}
    for split in SPLIT_FILES:
        split_rows[split], dataset_hashes[split] = load_split(split)
    eval_rows, selected_retain_groups, collision_row_indices = choose_eval_rows(split_rows)

    tokenizer = AutoTokenizer.from_pretrained(
        spec["repo_id"],
        revision=spec["revision"],
        cache_dir=str(model_cache),
        use_fast=True,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    if not tokenizer.is_fast:
        raise RuntimeError("Fast tokenizer required for this pinned scoring implementation")
    encoded_rows = [encode_like_open_unlearning(tokenizer, row["question"], row["answer"]) for row in eval_rows]
    input_sequence_lengths = [len(input_ids) for input_ids, _ in encoded_rows]
    model = AutoModelForCausalLM.from_pretrained(
        spec["repo_id"],
        revision=spec["revision"],
        cache_dir=str(model_cache),
        dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto",
        max_memory={0: "12GiB", 1: "12GiB", "cpu": "40GiB"},
        attn_implementation="sdpa",
    )
    model.eval()
    model.config.use_cache = False
    actual_batch_size = batch_size
    scored = answer_probabilities(model, encoded_rows, int(tokenizer.pad_token_id), actual_batch_size)
    if len(scored) != len(eval_rows):
        raise RuntimeError(f"Expected {len(eval_rows)} score rows; got {len(scored)}")

    score_rows = []
    for row, score in zip(eval_rows, scored):
        score_rows.append(
            {
                "split": row["split"],
                "row_index": row["row_index"],
                "profile_block": row["profile_block"],
                "question_sha256": hashlib.sha256(row["question"].encode("utf-8")).hexdigest(),
                "ambiguous_question_collision": row["split"] == "retain90" and row["row_index"] in collision_row_indices,
                "input_sequence_tokens": len(encoded_rows[len(score_rows)][0]),
                "avg_answer_token_loss": score["avg_answer_token_loss"],
                "answer_probability": score["answer_probability"],
                "scored_answer_tokens": score["scored_answer_tokens"],
            }
        )
    output = {
        "status": "PASS_BASELINE_DIAGNOSTIC",
        "model": spec,
        "model_config": {
            "model_type": model.config.model_type,
            "hidden_size": model.config.hidden_size,
            "num_hidden_layers": model.config.num_hidden_layers,
            "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
            "dtype": str(next(model.parameters()).dtype),
            "device_map": getattr(model, "hf_device_map", None),
        },
        "scoring": {
            "metric": "mean per-example exp(-mean assistant-token cross-entropy)",
            "metric_implementation": "Open-Unlearning evaluate_probability at pinned source revision",
            "dataset_revision": DATASET_REVISION,
            "dataset_sha256_by_split": dataset_hashes,
            "system_prompt": SYSTEM_PROMPT,
            "date_string": TEMPLATE_DATE,
            "declared_max_length": MAX_LENGTH,
            "input_sequences_truncated": False,
            "profile_block_size": PROFILE_QA_COUNT,
            "retain_profile_sample_seed": PROFILE_SAMPLE_SEED,
            "retain_profile_blocks_selected": selected_retain_groups,
            "retain_ambiguous_question_collision_rows_in_sample": collision_row_indices,
            "samples_per_split": {split: sum(r["split"] == split for r in score_rows) for split in SPLIT_FILES},
            "max_input_sequence_tokens": max(input_sequence_lengths),
            "sequences_over_declared_max_length": sum(length > MAX_LENGTH for length in input_sequence_lengths),
            "all_rows_have_scored_answer_tokens": all(row["scored_answer_tokens"] > 0 for row in score_rows),
        },
        "summary": summarize_scores(score_rows),
        "scores": score_rows,
        "runtime": {
            "torch_version": torch.__version__,
            "transformers_version": transformers_version,
            "accelerate_version": accelerate_version,
            "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "gpu_total_memory_bytes": [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())],
            "elapsed_seconds": round(time.monotonic() - started, 2),
            "training_started": False,
            "research_adapter_created": False,
        },
    }
    output_path = output_dir / f"{model_key}_baseline.json"
    output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    score_path = output_dir / f"{model_key}_scores.jsonl"
    score_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in score_rows), encoding="utf-8")

    del model, tokenizer, encoded_rows
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(model_cache, ignore_errors=True)
    return output


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output-dir", type=Path, default=Path("/kaggle/working/tofu_baseline_results"))
    parser.add_argument("--models", default="llama_3_2_3b,llama_3_1_8b")
    parser.add_argument("--batch-size", type=int, default=2)
    args = parser.parse_args()
    selected_models = [item.strip() for item in args.models.split(",") if item.strip()]
    unknown = set(selected_models) - set(MODELS)
    if unknown:
        parser.error(f"Unknown models: {sorted(unknown)}")
    if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
        raise RuntimeError("This bounded diagnostic requires Kaggle GPU T4 x2; no CUDA model run was started")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    status = {
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "selected_models": selected_models,
        "model_results": {},
        "training_started": False,
    }
    status_path = args.output_dir / "run_status.json"
    status_path.write_text(json.dumps(status, indent=2) + "\n", encoding="utf-8")
    for model_key in selected_models:
        try:
            result = evaluate_model(model_key, args.output_dir, args.batch_size)
            status["model_results"][model_key] = {
                "status": result["status"],
                "summary": result["summary"],
                "result_file": f"{model_key}_baseline.json",
                "scores_file": f"{model_key}_scores.jsonl",
            }
        except Exception as error:
            status["model_results"][model_key] = {
                "status": "ERROR",
                "error_type": type(error).__name__,
                "error": str(error),
            }
            status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
            raise
        status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        print(f"{model_key}: PASS_BASELINE_DIAGNOSTIC")
        print(json.dumps(result["summary"], indent=2))
    status["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
    status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"Results saved under: {args.output_dir}")


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
OUT = Path('/kaggle/working/tofu_baseline_results')
OUT.mkdir(parents=True, exist_ok=True)
!python /kaggle/working/diagnose_tofu.py --output-dir /kaggle/working/tofu_baseline_results/audit

In [ ]:
import subprocess, sys
run = subprocess.run([sys.executable, '/kaggle/working/tofu_baseline_eval.py', '--output-dir', '/kaggle/working/tofu_baseline_results', '--models', 'llama_3_2_3b,llama_3_1_8b', '--batch-size', '2'])
subprocess.run(['bash', '-lc', 'cd /kaggle/working && zip -rq tofu_baseline_diagnostic_results.zip tofu_baseline_results'], check=True)
if run.returncode:
    raise RuntimeError(f'Baseline runner exited with status {run.returncode}; partial status/score files were zipped.')
print('Download /kaggle/working/tofu_baseline_diagnostic_results.zip from the notebook Output panel.')